## API exploration

Exploring the API of "https://hubeau.eaufrance.fr/api/v1/niveaux_nappes/stations" to assess feasibility of a ML project.

In [2]:
from ades import GestionnairePiezometrie

In [4]:

gestionnaire = GestionnairePiezometrie(dossier_sortie="data")
# Liste de vos forages (remplacez par les vrais codes de votre bassin versant)
forages_bassin = [
        "BSS001EUKK", # forage à Saffré
        "BSS001PYMM", # Mâcon (issu de la doc)
        "BSS001PYKW"  # Un autre exemple
    ]

df = gestionnaire.telecharger_bassin_versant(forages_bassin,
                                            pause_secondes=3 )

--- Début du traitement pour 3 forages ---
Téléchargement de BSS001EUKK...
  -> Succès : 2003 mesures sauvegardées dans piezo_BSS001EUKK.csv
  [Pause de 3s pour ne pas surcharger le serveur...]
Téléchargement de BSS001PYMM...
  -> Succès : 6542 mesures sauvegardées dans piezo_BSS001PYMM.csv
  [Pause de 3s pour ne pas surcharger le serveur...]
Téléchargement de BSS001PYKW...
  -> Succès : 1463 mesures sauvegardées dans piezo_BSS001PYKW.csv
--- Traitement du bassin versant terminé ! ---


AttributeError: 'NoneType' object has no attribute 'head'

### Data Engineering

**To do list**

- Creer un dataframe à partir du JSON. Refactoriser le code de la fonction en module
- Explorer les données vendéennes :
examples : CHallans BSS001LADS   Entité  121
        niort
        la roche s yon BSS001MHUZ  Entité
        Cholet   BSS001JWAE          Entité       181
          ruffec BSS001RRGC

- Essayer de charger la vendée en entier.
- Moyenne des piezos de vendée.
- Sortir les coordonnées géographiques Lambert 93

### Emplacement géographiques des différentes stations de mesure.

La liste des stations existentes est stockée différemment des données piezometrique. Une autre database JSON

In [ ]:
import requests

def rechercher_stations_piezo(code_dep=None, code_region=None, code_bdlisa=None):
    url_stations = "https://hubeau.eaufrance.fr/api/v1/niveaux_nappes/stations"

    parametres = {
        "format": "json",
        "size": 5000
    }

    if code_dep:
        parametres["code_departement"] = str(code_dep)
    if code_region:
        parametres["code_region"] = str(code_region)
    if code_bdlisa:
        parametres["codes_bdlisa"] = str(code_bdlisa)

    print(f"Recherche des stations en cours avec les paramètres : {parametres}...")

    try:
        reponse = requests.get(url_stations, params=parametres)
        reponse.raise_for_status()

        donnees = reponse.json()

        # Extraction des bss_id
        liste_stations = []
        if "data" in donnees:
            for station in donnees["data"]:
                # bien récupérer le code BSS
                if "bss_id" in station and station["bss_id"]:
                    liste_stations.append(station["bss_id"])

        print(f"-> {len(liste_stations)} stations trouvées !")
        return liste_stations

    except requests.exceptions.RequestException as e:
        print(f"Erreur lors de la communication avec l'API : {e}")
        return []


In [ ]:
print("--- Recherche par département ---")
stations_du_44 = rechercher_stations_piezo(code_dep="44")
print(f"\n{stations_du_44[:]}")

print("\n--- Recherche par entité hydrogéologique ---")
stations_meme_nappe = rechercher_stations_piezo(code_bdlisa="113AC10")
print(f"\nVoici les 5 premières stations de la nappe : {stations_meme_nappe[:5]}")

staticme

--- Recherche par département ---
Recherche des stations en cours avec les paramètres : {'format': 'json', 'size': 5000, 'code_departement': '44'}...
-> 58 stations trouvées !

['BSS001EUMW', 'BSS001GPCB', 'BSS001EUZL', 'BSS001JPMN', 'BSS001EUJD', 'BSS001ETCD', 'BSS001DJDQ', 'BSS001DJDC', 'BSS001ETCE', 'BSS001DKCX', 'BSS001JRHS', 'BSS001HBQB', 'BSS001EUZK', 'BSS001JPMP', 'BSS001HBQA', 'BSS001DMWC', 'BSS001JPMQ', 'BSS001EUHC', 'BSS001DJDP', 'BSS001EUKK', 'BSS001ESVX', 'BSS001ESVY', 'BSS001EUZH', 'BSS001DKCW', 'BSS001EUPV', 'BSS001JSBK', 'BSS001DMWB', 'BSS001DLRR', 'BSS001DMWF', 'BSS001ESHE', 'BSS001GNWX', 'BSS001BLTP', 'BSS001GPPU', 'BSS001GSMF', 'BSS001GPZF', 'BSS001JTSD', 'BSS001JSNX', 'BSS001EUNG', 'BSS004DJRC', 'BSS003XVPE', 'BSS004DJRF', 'BSS004DJRG', 'BSS004DJRJ', 'BSS004DJRM', 'BSS004DJRQ', 'BSS003XVQS', 'BSS004DJRS', 'BSS004DJRT', 'BSS003ZKDU', 'BSS004DJQU', 'BSS004DJRU', 'BSS004DJRV', 'BSS004DJQW', 'BSS003XVPY', 'BSS001JQAJ', 'BSS001JRKS', 'BSS001JTRR', 'BSS001JNYB']

--- Reche

### Plots et un peu d'exploration de données. 


- plot dans le temps

- comparaison des données entre deux piezometres

- valeurs moyennes sur plusieurs entitées

- geopandas pour une representation graphique